In [0]:
from pyspark.sql import functions as F

# Шляхи до даних
vol_path = "/Volumes/db_workspace_olena/default/my_data/"

# Читання всіх файлів
flights_df = spark.read.csv(vol_path + "flights.csv", header=True, inferSchema=True)
airports_df = spark.read.csv(vol_path + "airports.csv", header=True, inferSchema=True)
airlines_df = spark.read.csv(vol_path + "airlines.csv", header=True, inferSchema=True)

# ПРЕПРОЦЕСИНГ: Об'єднання (Join) та Фільтрація (31 січня)
jan_31_data = flights_df.filter((F.col("MONTH") == 1) & (F.col("DAY") == 31)) \
    .join(airlines_df.alias("air"), flights_df["AIRLINE"] == F.col("air.IATA_CODE"), "left") \
    .join(airports_df.alias("org"), flights_df["ORIGIN_AIRPORT"] == F.col("org.IATA_CODE"), "left") \
    .join(airports_df.alias("dst"), flights_df["DESTINATION_AIRPORT"] == F.col("dst.IATA_CODE"), "left") \
    .select(
        flights_df["AIRLINE"].alias("airline_code"), # Явно беремо з flights_df
        F.col("air.AIRLINE").alias("airline"),       # Явно беремо з аліасу air (назва компанії)
        F.concat(F.lit("2015-01-"), F.col("DAY")).alias("flight_date"), 
        F.col("org.STATE").alias("origin_state"),
        F.col("dst.STATE").alias("destination_state"),
        F.col("DEPARTURE_DELAY").alias("departure_delay"),
        F.col("ARRIVAL_DELAY").alias("arrival_delay"),
        F.col("CANCELLED").alias("cancellation")
    )

# Перевірка результату
count_rows = jan_31_data.count()
print(f"Підготовлено рядків для Cosmos DB: {count_rows}")
display(jan_31_data.limit(10))

# Збереження результату 
jan_31_data.coalesce(1).write.mode("overwrite").option("header", "true").csv(vol_path + "output/cosmos_upload_data")

print("Файл успішно збережено у Volume за шляхом output/cosmos_upload_data")

Підготовлено рядків для Cosmos DB: 12037


airline_code,airline,flight_date,origin_state,destination_state,departure_delay,arrival_delay,cancellation
AA,American Airlines Inc.,2015-01-31,CA,TX,-3,-2,0
US,US Airways Inc.,2015-01-31,AZ,NC,23,10,0
AA,American Airlines Inc.,2015-01-31,CA,FL,-3,14,0
NK,Spirit Air Lines,2015-01-31,NV,MN,-5,-23,0
US,US Airways Inc.,2015-01-31,UT,NC,6,-10,0
US,US Airways Inc.,2015-01-31,CA,PA,-4,-13,0
AA,American Airlines Inc.,2015-01-31,CA,TX,-5,7,0
DL,Delta Air Lines Inc.,2015-01-31,CA,MN,-5,5,0
US,US Airways Inc.,2015-01-31,CA,NC,-2,18,0
DL,Delta Air Lines Inc.,2015-01-31,AK,WA,-7,-35,0


Файл успішно збережено у Volume за шляхом output/cosmos_upload_data
